# Clase 155 — LLM Evaluation: MMLU, MT-Bench, LLM-as-judge

Simulamos MMLU, MT-Bench y un LLM-as-judge sintético. Discutimos biases comunes del judge.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1. MMLU mini (20 preguntas multiple-choice sintéticas)

4 categorías × 5 preguntas. "Model" responde con accuracy 60% sesgada por categoría.

In [ ]:
categories = ['math', 'history', 'biology', 'code']
mmlu = []
for cat in categories:
    for i in range(5):
        correct = rng.integers(0, 4)
        mmlu.append({'cat': cat, 'q': f'{cat} Q{i}', 'choices': list('ABCD'), 'gold': correct})
print(f'MMLU mini: {len(mmlu)} preguntas, {len(categories)} categorías')

In [ ]:
# Model accuracy variable por categoría
model_acc = {'math': 0.4, 'history': 0.7, 'biology': 0.65, 'code': 0.55}

def fake_model(question):
    p = model_acc[question['cat']]
    if rng.random() < p: return question['gold']
    # respuesta incorrecta random entre las otras 3
    wrong = [i for i in range(4) if i != question['gold']]
    return rng.choice(wrong)

preds = [fake_model(q) for q in mmlu]
from collections import defaultdict
per_cat = defaultdict(list)
for q, p in zip(mmlu, preds): per_cat[q['cat']].append(int(p == q['gold']))
print('Accuracy por categoría:')
for cat, vals in per_cat.items():
    print(f'  {cat:8s}: {np.mean(vals):.2%} ({sum(vals)}/{len(vals)})')
overall = np.mean([int(p == q['gold']) for q, p in zip(mmlu, preds)])
print(f'\noverall MMLU: {overall:.2%}')

## 2. MT-Bench: 2 turnos (follow-up tests reasoning)

In [ ]:
mtbench = [
    {'t1': 'Explicá qué es overfitting.', 't2': 'Dame un ejemplo con árboles de decisión.'},
    {'t1': 'Resumí RAG en 1 oración.', 't2': '¿Cuándo NO usar RAG?'},
    {'t1': 'Diferencia LoRA vs full FT.', 't2': '¿Qué pasa si r es muy chico?'},
]

def fake_response(prompt, turn):
    qual = rng.uniform(0.5, 1.0) - 0.1 * (turn == 2)   # turn 2 suele ser peor
    return f'[resp turn{turn} q={qual:.2f}] ' + 'X' * int(30 * qual)

def mtbench_score(response):
    """Score 1-10 heurístico (longitud + variabilidad)."""
    return min(10, max(1, int(len(response) / 5)))

scores = []
for ex in mtbench:
    r1 = fake_response(ex['t1'], 1); s1 = mtbench_score(r1)
    r2 = fake_response(ex['t2'], 2); s2 = mtbench_score(r2)
    scores.append((s1, s2))
    print(f"Q: {ex['t1'][:35]:35s} | turn1={s1} | turn2={s2}")
print(f"\nMT-Bench avg: turn1={np.mean([s[0] for s in scores]):.2f} | turn2={np.mean([s[1] for s in scores]):.2f}")

## 3. LLM-as-judge (A vs B)

In [ ]:
GOOD_KEYS = {'porque', 'ejemplo', 'es decir', 'en resumen', 'específicamente'}

def llm_judge(prompt, resp_a, resp_b):
    """Heurística: longitud + presencia de keywords + tie si diff < 5%."""
    def score(r):
        s = len(r)
        s += 10 * sum(k in r.lower() for k in GOOD_KEYS)
        return s
    sa, sb = score(resp_a), score(resp_b)
    if abs(sa - sb) / max(sa, sb, 1) < 0.05: return 'tie'
    return 'A' if sa > sb else 'B'

# 50 batallas A vs B con A mejor en promedio
wins = {'A': 0, 'B': 0, 'tie': 0}
for _ in range(50):
    ra = 'porque ' + 'X' * rng.integers(50, 150)
    rb = 'X' * rng.integers(50, 150)
    wins[llm_judge('Q', ra, rb)] += 1
print(f'A wins: {wins["A"]} | B wins: {wins["B"]} | ties: {wins["tie"]}')
win_rate_A = wins['A'] / (wins['A'] + wins['B'])
print(f'win-rate A: {win_rate_A:.1%}')

## 4. Biases del judge

- **Position bias**: prefiere la respuesta listada primero. Mitigación: aleatorizar y promediar.
- **Length bias**: prefiere respuestas largas (nuestra heurística lo demuestra).
- **Self-preference**: GPT-4 tiende a preferir respuestas de GPT-4.
- **Verbosity bias** ≈ length bias.
- Mitigación: prompt engineering del judge, multiple judges, calibración con humanos.

In [ ]:
# Position bias check: swap A/B y re-juzgar
consistent = 0
for _ in range(50):
    ra = 'porque ' + 'X' * rng.integers(50, 150)
    rb = 'X' * rng.integers(50, 150)
    v1 = llm_judge('Q', ra, rb)
    v2 = llm_judge('Q', rb, ra)
    swap = {'A': 'B', 'B': 'A', 'tie': 'tie'}
    if v1 == swap[v2]: consistent += 1
print(f'consistencia bajo swap: {consistent}/50 = {consistent/50:.1%}')
print('→ una heurística determinística da 100%; un LLM real ~70-85%, evidencia de position bias.')

## 5. ELO conceptual

Chatbot Arena agrega muchos pareos A-vs-B y calcula ELO como en ajedrez:

$$ R_A' = R_A + K \cdot (S_A - E_A), \quad E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}} $$

Con miles de pareos → ranking estable. Es la métrica de referencia 2024-2026 para chatbots.

In [ ]:
# Mini ELO: 3 modelos, 200 batallas
models = {'M1': 1500, 'M2': 1500, 'M3': 1500}
true_skill = {'M1': 1600, 'M2': 1500, 'M3': 1400}
K = 32
for _ in range(200):
    a, b = rng.choice(list(models), 2, replace=False)
    p_a = 1 / (1 + 10 ** ((true_skill[b] - true_skill[a]) / 400))
    s_a = 1 if rng.random() < p_a else 0
    e_a = 1 / (1 + 10 ** ((models[b] - models[a]) / 400))
    models[a] += K * (s_a - e_a); models[b] += K * ((1 - s_a) - (1 - e_a))
for m, r in sorted(models.items(), key=lambda x: -x[1]):
    print(f'  {m}: ELO={r:.0f} (true skill={true_skill[m]})')

print('\n## Ejercicio guiado')
print('1. Pedile al judge que ignore length explícitamente; medir cambio en win-rate.')
print('2. Implementar majority voting con 3 judges distintos.')
print('3. Calibrar judge con 20 batallas etiquetadas por humano (acc del judge vs gold).')

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — MMLU con lm-eval-harness (CLI)

In [ ]:
print('lm_eval --model hf '
      '--model_args pretrained=mistralai/Mistral-7B-v0.1 '
      '--tasks mmlu --num_fewshot 5 --batch_size 8')
print('Reporta accuracy por subtarea y promedio (57 materias, 4 opciones).')
# El notebook simula arriba el score agregado y por categoria.

### Ejercicio 2 — HumanEval: pass@k (núcleo numpy ejecutable)

In [ ]:
# pass@k insesgado (Chen et al. 2021): probabilidad de >=1 correcta en k muestras.
from math import comb
def pass_at_k(n, c, k):
    # n=muestras generadas, c=correctas, k=presupuesto
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)
assert pass_at_k(10, 5, 1) == 0.5           # 5/10 correctas -> pass@1 = 0.5
assert pass_at_k(5, 5, 1) == 1.0            # todas correctas -> 1.0
assert abs(pass_at_k(10, 1, 5) - (1 - comb(9, 5) / comb(10, 5))) < 1e-12
print('pass@1(10,5)=%.2f  pass@1(5,5)=%.2f  pass@5(10,1)=%.2f'
      % (pass_at_k(10, 5, 1), pass_at_k(5, 5, 1), pass_at_k(10, 1, 5)))
print('En real: generar n soluciones, ejecutar los unit tests, contar c.')

### Ejercicio 3 — MT-Bench con juez GPT-4 / Claude

In [ ]:
try:
    import anthropic  # noqa: F401
    _J = True
except Exception:
    _J = False
JUDGE = ('Actúa como juez imparcial. Puntúa la respuesta del asistente de 1 a 10 '
         'según utilidad, precisión y detalle. Devolvé solo el número.\n\n'
         'Pregunta: {q}\nRespuesta: {a}\nPuntaje:')
if _J:
    client = anthropic.Anthropic()
    # score = client.messages.create(model='claude-sonnet-4-5', max_tokens=8,
    #             messages=[{'role':'user','content': JUDGE.format(q=..., a=...)}])
    print('Juez LLM: prompt de puntaje 1-10 por turno; promediar los 80 items.')
else:
    print('Template de juez MT-Bench listo (puntaje 1-10, 2 turnos).')

### Ejercicio 4 — LLM-as-judge propio (A vs B, ejecutable)

In [ ]:
# Reutiliza llm_judge del notebook: A/B/tie sobre 20 pares.
import numpy as np
rng2 = np.random.default_rng(7)
wins = {'A': 0, 'B': 0, 'tie': 0}
for _ in range(20):
    ra = 'porque ' + 'x' * rng2.integers(80, 160)     # A con señal 'porque'
    rb = 'x' * rng2.integers(80, 160)
    wins[llm_judge('Q', ra, rb)] += 1
print('resultados:', wins)
assert wins['A'] >= wins['B']       # A (con keyword) gana en promedio

### Ejercicio 5 — Custom eval: 50 prompts + criterio de aceptación (ejecutable)

In [ ]:
import numpy as np
# Eval a medida: verifica que la salida cumpla un criterio duro por prompt.
cases = [{'out': 'La respuesta es 42.', 'must': '42'},
         {'out': 'El color es azul.',   'must': 'azul'},
         {'out': 'No lo sé.',           'must': 'No lo sé'}]
def acceptance(cases):
    return np.mean([c['must'] in c['out'] for c in cases])
score = acceptance(cases)
print(f'acceptance = {score:.0%}')
assert score == 1.0
print('En prod: 50 prompts de tu use case + criterios (regex/JSON/keyword/juez).')